# 리포트 18 — 가림 판정은 Sionna 광선엔진이 하고, 면적분은 우리 커널이 한다

> ### 한 일
> **상용 고주파 솔버의 순서 그대로 광선으로 조명면을 찾고 그 면 위에서 부품별 재질 PO 를 적분해 σ 를 냈다.**

### 결과
1. 첫 충돌 탐색과 가림은 Sionna 가 이미 들고 있는 Mitsuba/OptiX 엔진이 하고, 표면전류 적분과 σ 출력은 우리가 얹는다 — 그 문서에 `physical optics` 는 0 회 [^1] 나온다.
2. 조명원을 방위 280° [^2] · 고각 15° [^3] 에 두면 조명원을 향한 외피의 29 [^4]~47% [^5] 가 기체 자신에 가려 있다.
3. 그 가림을 끄면 방위평균 σ 가 최대 6.63 dB [^6] (Matrice 4E ⭐ [^7], 닫힌 동체)까지 부풀고, 열린 프레임인 S1000+ [^8] 에서는 0.11 dB [^9] 다.
4. 기체 7 종 [^10] 전부에서 이 값이 이산화 바닥(최대 0.071 dB [^11]) 위에 있다 — 가림은 수치잡음이 아니라 물리다.
5. 금속 4그룹만 남긴 메쉬의 방위평균 σ 가 전체의 112% [^12] 다 — 코히런트 합이라 100 % 를 넘는다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| ① 조명면 찾기 | Sionna 의 Mitsuba/OptiX 광선엔진을 그대로 부른다 — 첫 충돌 탐색과 자기가림 판정이 그쪽 몫이다 |
| ② 면적분 | 그 면 위에서 부품별 재질 PO 를 적분한다 (`src/rcs_sbr.py` `rcs_sbr()`) — E = Σ \|Γᵢ(θᵢ)\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² |
| 셸 투과 | 얇은 유전체 셸 뒤의 금속(배터리·PCB)을 코히런트 합산한다 (동 `penetrate=True`) |
| 가림의 크기 | 같은 자세에서 가림을 끄고 다시 적분해 방위평균 σ 의 차이를 기체마다 잰다 — 이산화 바닥과 나란히 싣는다 |

### 재현

```bash
PYTHONPATH=src python src/make_report02_target.py --derive-only
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/report02_derived.json`, `outputs/prior_settled_sionna.json`, `outputs/report3_rt.json` |
| 소요 | 약 2분 (GPU 0장 — 원장 조립이다) |
| 비고 | σ 격자 자체의 재생성은 `benchmark/rcs_anchor.py` 가 맡는다 |

---

## 두 낱말을 먼저 푼다

**PO** 는 물리광학(physical optics)이다 — 빛이 닿는 면에 흐르는 전류를 근사식으로 바로 적어 넣고 그 면을 훑어 더해 산란을 내는 방법이다. **SBR** 은 광선을 쏴서 튀기며 그 면이 어디인지 찾는 방법(shooting-and-bouncing rays)이다.

상용 고주파 RCS 솔버(FEKO/CST SBR+)의 순서 그대로다 — **① 광선으로 실제 조명면을 찾고 ② 그 위에서 PO 표면적분**(`src/rcs_sbr.py` `rcs_sbr()`). 레이다식이 표적 산란과 전파 경로를 두 양으로 쓰는 그대로, **σ 는 이 커널이 내고 경로와 환경은 그 엔진이 낸다**.

## 누가 무엇을 하나

| 단계 | 무엇을 | 누가 |
|---|---|---|
| 첫 충돌 탐색 · 가림 | 어느 면이 실제로 조명되는가 | 🟢 Sionna 의 Mitsuba/OptiX 광선엔진 |
| 재질 \|Γ(θ)\| | 수직입사 보정값 × 각도 모양(TE·TM 전력평균, `ANGLE_GAMMA=1` 기본) | 🟢 Sionna 재질표(`src/materials.py` `MATERIALS`) + 🔵 각도 모양 (`src/rcs_sbr.py` `ANGLE_GAMMA`) |
| PO 면적분 → σ | E = Σ \|Γᵢ(θᵢ)\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² | 🔵 우리 (`src/rcs_sbr.py` `rcs_sbr()`) |
| 셸 투과 | 얇은 유전체 셸 뒤 금속(배터리·PCB)의 코히런트 합 | 🔵 우리 (동 `penetrate=True`) |

## 왜 우리가 얹어야 하나

Sionna 는 광선을 쏘고 튀긴다 — 기술보고서(v1.2, 59쪽)에 SBR 이 48 회 [^13] 나오고 우리도 그 엔진을 그대로 부른다. 같은 문서에서 `physical optics` 0 회 [^1] · `radar cross section` 0 회 [^14] · `surface current` 0 회 [^15] 이고, 거친 면은 정규화 산란패턴을 쓰는 경험 모델이다 — 그 셈이 어디서 끝나는지는 [부 1 «스톡 엔진이 하는 일과 안 하는 일»](../README.md#부-1-스톡-엔진이-하는-일과-안-하는-일) 가 인자 목록까지 해부했다.

ITU `metal` 의 산란계수 S = 0.0 [^16] 이라 스톡 산란 모델이 금속에서 내놓는 항은 0 이고, 우리 σ 는 면적분에서 창발한다. 금속 4그룹(모터·배터리·PCB·카메라)만 남긴 메쉬의 방위평균 σ 는 전체의 112% [^12] 다.

## PO 적분이 실제로 올라타는 면은 어디까지인가

![mesh_compare_material_shadow](../outputs/figures/mesh_compare_material_shadow.png)

**그림 1.** PO 적분이 실제로 올라타는 면은 어디까지인가?

조명원을 방위 280° [^2] · 고각 15° [^3] 에 두었다 — 방위 72 점 [^17] 스윕에서 7기체 평균 그늘비율의 중앙값에 가장 가까운 방위이고, 규칙이 고른다. 가림 판정은 생산 SBR 이 쓰는 그림자광선 그대로다(`rcs_sbr._exit_visible()`). 그림의 색은 재질이 아니라 조명 상태다.

| 기체 | 외피 그늘 | 가림 [dB] | 셸 투과 [dB] | 합 [dB] | 이산화 바닥 [dB] | 생산 σ [dBsm] |
|---|---|---|---|---|---|---|
| Mini 5 Pro ⭐ | 41 % | +5.98 | +3.66 | +2.32 | 0.014 | -22.0 |
| Mavic 4 Pro | 29 % | +3.75 | +2.55 | +1.20 | 0.021 | -18.2 |
| Matrice 4E ⭐ | 35 % | +6.63 | +3.72 | +2.90 | 0.021 | -18.9 |
| Phantom 4 | 36 % | +5.90 | +4.06 | +1.84 | 0.056 | -19.9 |
| X500 V2 | 47 % | +1.07 | +0.00 | +1.07 | 0.037 | -16.8 |
| Typhoon H (H480) | 39 % | +2.91 | +1.51 | +1.40 | 0.014 | -15.8 |
| S1000+ | 42 % | +0.11 | -0.19 | +0.30 | 0.071 | -12.3 |

출처 [^18]

## 가림을 끄면 얼마나 부푸나

가림을 끄면 방위평균 σ 가 6.63 dB [^6] (Matrice 4E ⭐ [^7], 닫힌 동체)까지 부풀고, 열린 프레임인 S1000+ [^8] 에서는 0.11 dB [^9] 다. 기체 7 종 [^10] 전부에서 이 값이 이산화 바닥(최대 0.071 dB [^11]) 위에 있다.

⚠ 이 표와 그림은 2026-08-04 [^19] 형상 정정 **전** 메쉬 기준이고, 2026-08-07 10:58:22 [^20] Γ(θ) 각도 모양(기본 켬) **이전** 커널의 산출이다 — 가림 최대치를 내는 Matrice 4E 와 X500 V2 가 그 정정을 받은 기체이고, 닫힌 동체의 가림은 셸 형상에 직접 걸린다. 생산 σ 열도 두 축 같은 이유로 재계산 대상이다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 정정된 메쉬로 가림 표와 생산 σ 를 같은 설정에서 다시 낸다 | 형상 정정이 가림과 σ 를 어느 방향으로 얼마나 옮기는지가 기체별로 확정된다 | [^21] |
| 같은 메쉬를 스톡 경로 솔버에 그대로 넣고 무엇이 나오는지 잰다 | 우리 커널이 스톡 위에 얹은 항이 무엇인지가 나란히 확정된다 | [편 19 «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…»](19_kernel-vs-stock.ipynb) |
| 수신 방향 그림자 광선을 켜고 바이스태틱으로 넓힌다 | 출사 쪽 가림이 상반성 위반을 얼마나 줄이는지가 확정된다 | [편 20 «수신 방향 그림자 광선을 켜면 상반성 위반이…»](20_bistatic-exit.ipynb) |
| PO 면적분을 디바이스 커널로 옮긴다 | 전격자 재생성 비용이 확정된다 — 지금은 호스트가 대부분을 쓴다 | [편 19 «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…»](19_kernel-vs-stock.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 21개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.physical optics` | 0 |
| [^2] | `outputs/report02_derived.json` | `occlusion.az_deg` | 280 |
| [^3] | `outputs/report02_derived.json` | `occlusion.el_deg` | 15 |
| [^4] | `outputs/report02_derived.json` | `occlusion.shadow_min_pct` | 29.06 |
| [^5] | `outputs/report02_derived.json` | `occlusion.shadow_max_pct` | 46.54 |
| [^6] | `outputs/report02_derived.json` | `occlusion.max_db` | 6.626 |
| [^7] | `outputs/report02_derived.json` | `occlusion.max_drone` | Matrice 4E ⭐ |
| [^8] | `outputs/report02_derived.json` | `occlusion.min_drone` | S1000+ |
| [^9] | `outputs/report02_derived.json` | `occlusion.min_db` | 0.1111 |
| [^10] | `outputs/report02_derived.json` | `occlusion.n_above_floor` | 7 |
| [^11] | `outputs/report02_derived.json` | `occlusion.floor_max_db` | 0.07061 |
| [^12] | `outputs/report3_rt.json` | `C_metal.metal_share_pct` | 112 |
| [^13] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.SBR or shooting-and-bouncing` | 48 |
| [^14] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.radar cross section` | 0 |
| [^15] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.surface current` | 0 |
| [^16] | `outputs/report3_rt.json` | `C_metal.itu_metal_S` | 0 |
| [^17] | `outputs/report02_derived.json` | `occlusion.n_az_sweep` | 72 |
| [^18] | `outputs/report02_derived.json` | `occlusion.rows` | (7행 표) |
| [^19] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^20] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^21] | `outputs/meshfix_attack.json` | `recommended_gate_before_any_sigma_claim` | (6행 표) |